In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from src.train_logistic import test_models,get_best_model,predict_new_labels

import os 
os.getcwd()

'c:\\Users\\estev\\Documents\\M1_MIASHS\\DemiJourDS1\\notebooks'

## TRAINING — RÉGRESSION LOGISTIQUE

Dans le dossier `src`, nous avons implémenté l'ensemble de la pipeline utilisée pour l'entraînement du modèle de régression logistique.

Nous nous intéressons ici à l'optimisation de trois paramètres principaux :

1. **Le nombre de variables utilisées** : la dimension initiale de nos données étant de ${R}^7$, nous cherchons à étudier l'influence de différents sous-ensembles de variables sur les performances du modèle.

2. **Le paramètre $\alpha$** : il contrôle la régularisation du modèle. La régularisation permet de limiter la complexité du modèle afin de réduire le risque de surapprentissage (*overfitting*).

$$
\min_{\beta} \left[
-\frac{1}{n}\sum_{i=1}^{n}
\left(
y_i\log(p_i) + (1-y_i)\log(1-p_i)
\right)
+
\frac{\alpha}{2}\|\beta\|_2^2
\right]
$$

où $p_i$ représente la probabilité prédite par le modèle pour l'observation $i$.

Lorsque $\alpha$ augmente, la pénalisation des coefficients devient plus importante, ce qui contraint davantage le modèle. À l'inverse, une faible valeur de $\alpha$ permet au modèle de s'adapter davantage aux données d'entraînement.

3. **Le seuil de classification** : celui-ci permet de transformer la probabilité prédite par le modèle en une classe. Par défaut, un seuil de $0.5$ est généralement utilisé :

$$
\hat{y}_i =
\begin{cases}
1 & \text{si } p_i \geq 0.5,\\
0 & \text{sinon}.
\end{cases}
$$

Nous chercherons également à étudier l'influence de ce seuil sur les performances du modèle, même si notre jeu de données contient des classes équilibrées.

### Standardisation des données

Nous allons tout d'abord appliquer une **standardisation par score-z** : 

$$
Z_j = \frac{X_j-\mu_j}{\sigma_j}
$$




In [2]:
res_std = test_models(train_path="../data/farms_train.csv",test_path = "../data/farms_test.csv",z_score=True)

Top 10 des meilleures combinaisons (n_vars, alpha, threshold) par AUC puis F1 :

 n_vars     alpha  threshold    auc       f1  accuracy
      7 31.622777       0.50 0.8934 0.840095    0.8325
      7 31.622777       0.40 0.8934 0.830769    0.8075
      7 31.622777       0.45 0.8934 0.829493    0.8150
      7 31.622777       0.55 0.8934 0.817259    0.8200
      7 31.622777       0.35 0.8934 0.800000    0.7625
      7 31.622777       0.30 0.8934 0.780000    0.7250
      7 31.622777       0.60 0.8934 0.768392    0.7875
      7 31.622777       0.25 0.8934 0.757692    0.6850
      7 31.622777       0.20 0.8934 0.737037    0.6450
      7 31.622777       0.65 0.8934 0.710059    0.7550

Meilleure config (AUC) : n_vars=7, alpha=31.6228, threshold=0.5 -> AUC = 0.8934, F1 = 0.8401
Meilleure config (F1)  : n_vars=7, alpha=3.1623, threshold=0.4 -> AUC = 0.8920, F1 = 0.8433




Nous allons tout d'abord appliquer une **standardisation Min-Max** :

$$
Z_j = \frac{X_j-X_{j,\min}}{X_{j,\max}-X_{j,\min}}
$$

Cette transformation permet de ramener les valeurs de chaque variable dans l'intervalle :

$$
Z_j \in [0,1]
$$

In [3]:
res_min_max = test_models(train_path="../data/farms_train.csv",test_path = "../data/farms_test.csv",z_score=False,min_max=True)

Top 10 des meilleures combinaisons (n_vars, alpha, threshold) par AUC puis F1 :

 n_vars  alpha  threshold      auc       f1  accuracy
      7    1.0       0.45 0.893575 0.837529    0.8225
      7    1.0       0.50 0.893575 0.837321    0.8300
      7    1.0       0.40 0.893575 0.832599    0.8100
      7    1.0       0.55 0.893575 0.823232    0.8250
      7    1.0       0.35 0.893575 0.800847    0.7650
      7    1.0       0.30 0.893575 0.789474    0.7400
      7    1.0       0.60 0.893575 0.771739    0.7900
      7    1.0       0.25 0.893575 0.760618    0.6900
      7    1.0       0.20 0.893575 0.738404    0.6475
      7    1.0       0.65 0.893575 0.717647    0.7600

Meilleure config (AUC) : n_vars=7, alpha=1.0000, threshold=0.45 -> AUC = 0.8936, F1 = 0.8375
Meilleure config (F1)  : n_vars=7, alpha=3.1623, threshold=0.5 -> AUC = 0.8932, F1 = 0.8443


Pour finir, nous testons le modèle sans aucune standardisation.


In [4]:
res_sans_std = test_models(train_path="../data/farms_train.csv",test_path = "../data/farms_test.csv",z_score=False,min_max=False)

Top 10 des meilleures combinaisons (n_vars, alpha, threshold) par AUC puis F1 :

 n_vars  alpha  threshold     auc       f1  accuracy
      7   0.01       0.45 0.89215 0.844869    0.8375
      7   0.01       0.40 0.89215 0.834862    0.8200
      7   0.01       0.35 0.89215 0.829596    0.8100
      7   0.01       0.50 0.89215 0.826733    0.8250
      7   0.01       0.30 0.89215 0.823789    0.8000
      7   0.01       0.55 0.89215 0.814070    0.8150
      7   0.01       0.25 0.89215 0.805970    0.7725
      7   0.01       0.60 0.89215 0.803150    0.8125
      7   0.01       0.20 0.89215 0.794179    0.7525
      7   0.01       0.65 0.89215 0.783784    0.8000

Meilleure config (AUC) : n_vars=7, alpha=0.0100, threshold=0.45 -> AUC = 0.8921, F1 = 0.8449
Meilleure config (F1)  : n_vars=7, alpha=0.0100, threshold=0.45 -> AUC = 0.8921, F1 = 0.8449


Le choix de la standardisation n'a pas beaucoup d'influence sur les performances du modèle. Néanmoins, on constate que le seuil moyen le plus utilisé est de $0.5$. Cela fait sens puisque les données sont parfaitement équilibrées.

En revanche, la valeur optimale de $\alpha$ varie selon que les données sont standardisées ou non. Cela peut s'expliquer par le fait que la régularisation dépend de l'échelle des variables : lorsque les variables sont déjà dans des ordres de grandeur similaires, la standardisation a moins d'impact sur le modèle.

Ainsi, la standardisation ne semble pas être un prérequis dans notre cas. Nous choisissons donc le modèle sans standardisation, avec un seuil de $0.45$ et $\alpha$ = $0.01$.
 

In [ ]:
test = get_best_model(train_path="../data/farms_train.csv",test_path = "../data/farms_test.csv",z_score=False,min_max=False)

predict_new_labels(test["model"],test["variables"],test["threshold"],
                   train_path="../data/farms_train.csv",test_path = "../data/farms_test.csv",res_path= "../res/resultat_logistic.csv",
                   z_score=False,min_max=False)

Top 10 des meilleures combinaisons (n_vars, alpha, threshold) par AUC puis F1 :

 n_vars  alpha  threshold     auc       f1  accuracy
      7   0.01       0.45 0.89215 0.844869    0.8375
      7   0.01       0.40 0.89215 0.834862    0.8200
      7   0.01       0.35 0.89215 0.829596    0.8100
      7   0.01       0.50 0.89215 0.826733    0.8250
      7   0.01       0.30 0.89215 0.823789    0.8000
      7   0.01       0.55 0.89215 0.814070    0.8150
      7   0.01       0.25 0.89215 0.805970    0.7725
      7   0.01       0.60 0.89215 0.803150    0.8125
      7   0.01       0.20 0.89215 0.794179    0.7525
      7   0.01       0.65 0.89215 0.783784    0.8000

Meilleure config (AUC) : n_vars=7, alpha=0.0100, threshold=0.45 -> AUC = 0.8921, F1 = 0.8449
Meilleure config (F1)  : n_vars=7, alpha=0.0100, threshold=0.45 -> AUC = 0.8921, F1 = 0.8449
Meilleur modèle sélectionné par 'auc' :
  variables  = ['TOF', 'AGE', 'R7', 'R8', 'R17', 'R22', 'R32']
  alpha      = 0.0100
  threshold  = 0.45
  AU

FileNotFoundError: [Errno 2] No such file or directory: 'data/farms_train.csv'